# Tool-Calling Research Agent (Teacher)

**Phase 1** of the BBB conference demo pipeline.

This notebook implements a simple tool-calling agent using the OpenAI API.
The agent researches a given stock ticker by calling financial tools (yfinance),
then produces a structured research memo.

The full conversation trajectory (including all tool calls and results) is saved
as training data for fine-tuning a smaller open-source model (Qwen3-4B).

## Architecture
```
User prompt → GPT (teacher) → tool_call → execute tool → result → GPT → ... → final memo
                                ↑                                        |
                                └── full trajectory saved as JSONL ──────┘
```

## Setup

In [9]:
import json
import os
import sys
from pathlib import Path

from openai import OpenAI
from dotenv import load_dotenv

# Add project root to path so we can import tools
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from tools.stock_tools import (
    TOOL_SCHEMAS,
    TOOL_FUNCTIONS,
    run_tool_calling_agent,
)

load_dotenv(PROJECT_ROOT / ".env")

client = OpenAI(base_url="https://us.api.openai.com/v1")

# Teacher model — change to gpt-5.4 when available
MODEL = "gpt-5.4"

## System Prompt

Instructs the model to:
1. Act as an equity research analyst
2. Use tools to gather data
3. Think before each tool call (in `<think>` tags)
4. Produce a structured research memo as final output

In [10]:
SYSTEM_PROMPT = """\
You are an equity research analyst conducting comprehensive research on a target company.

## Instructions
- Use the available tools to gather financial data, news, price history, and analyst recommendations.
- Before each tool call, briefly explain your reasoning in <think> tags.
- After gathering sufficient data, synthesize your findings into a structured research memo.
- Be thorough but efficient — aim for 3-5 tool calls total.

## Output Format
Your final response (after all tool calls) should be a markdown research memo with these sections:
- **Company Overview**: Brief description and market position
- **Financial Analysis**: Revenue, profitability, balance sheet highlights
- **Market Performance**: Price trends, volatility, trading patterns
- **Analyst Sentiment**: Consensus recommendations, recent upgrades/downgrades
- **Key Risks & Opportunities**: Summary of bull/bear case

## Important
- Use ONLY the tools provided. Do not make up financial data.
- Present facts objectively. Do not give investment advice.
- If a tool returns an error, note it and move on.
"""

## Tool Schemas

These are defined in `tools/stock_tools.py` and shared across all BBB notebooks.

In [11]:
# Quick look at what tools the agent has access to
for tool in TOOL_SCHEMAS:
    fn = tool["function"]
    params = list(fn["parameters"]["properties"].keys())
    print(f"  {fn['name']}({', '.join(params)})")
    print(f"    → {fn['description']}\n")

  get_stock_news(ticker)
    → Get the 5 most recent news articles for a stock ticker from Yahoo Finance.

  get_financials(ticker, statement_type, period)
    → Get financial statements (income statement, balance sheet, or cash flow) for a stock.

  get_price_history(ticker, period, interval)
    → Get historical stock price data (OHLCV) with summary statistics.

  get_recommendations(ticker, months_back)
    → Get analyst recommendations and recent upgrades/downgrades for a stock.



## Run the Agent

Let's test with a single ticker first.

In [12]:
trajectory = run_tool_calling_agent(
    client=client,
    model=MODEL,
    user_prompt="Research NVDA focusing on financial health and growth potential.",
    system_prompt=SYSTEM_PROMPT,
)

print(f"\nTrajectory has {len(trajectory)} messages")
print(f"Tool calls: {sum(1 for m in trajectory if m.get('role') == 'tool')}")

AuthenticationError: Error code: 401 - {'error': {'message': 'Attempted to access resource with incorrect regional hostname. Please make your request to us.api.openai.com', 'type': 'invalid_request_error', 'param': None, 'code': 'incorrect_hostname'}}

In [8]:
trajectory[2]

{'role': 'assistant',
 'content': '<think>To analyze NVDA\'s financial health and growth potential, I need to get a comprehensive view of its financial statements including income statement for profitability and revenue trends, and balance sheet for financial health. Additionally, recent news can provide insights into growth initiatives or challenges.</think>\n\nawait multi_tool_use.parallel({\n  tool_uses: [\n    {\n      recipient_name: "functions.get_financials",\n      parameters: { ticker: "NVDA", statement_type: "income", period: "annual" }\n    },\n    {\n      recipient_name: "functions.get_financials",\n      parameters: { ticker: "NVDA", statement_type: "balance_sheet", period: "annual" }\n    },\n    {\n      recipient_name: "functions.get_stock_news",\n      parameters: { ticker: "NVDA" }\n    }\n  ]\n})\n{\n  "tool_uses": [\n    {\n      "recipient_name": "functions.get_financials",\n      "parameters": { "ticker": "NVDA", "statement_type": "income", "period": "annual" },\

### Inspect the trajectory

In [ ]:
# Print each message role and a preview
for i, msg in enumerate(trajectory):
    role = msg["role"]
    content = msg.get("content", "")
    tool_calls = msg.get("tool_calls", [])
    
    if role == "system":
        print(f"[{i}] SYSTEM: {content[:80]}...")
    elif role == "user":
        print(f"[{i}] USER: {content}")
    elif role == "assistant" and tool_calls:
        names = [tc["function"]["name"] for tc in tool_calls]
        think = content[:100] if content else "(no content)"
        print(f"[{i}] ASSISTANT → calls {names}  |  {think}")
    elif role == "assistant":
        print(f"[{i}] ASSISTANT (final): {content[:120]}...")
    elif role == "tool":
        print(f"[{i}] TOOL ({msg.get('tool_call_id', '?')[:12]}): {content[:80]}...")

### View the final memo

In [ ]:
from IPython.display import Markdown, display

# The last assistant message (without tool_calls) is the final memo
final_msg = [m for m in trajectory if m["role"] == "assistant" and not m.get("tool_calls")][-1]
display(Markdown(final_msg["content"]))

## Save the Trajectory

Save in the format we'll use for training data.

In [ ]:
output_dir = PROJECT_ROOT / "data" / "bbb"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "tool_calling_trajectories.jsonl"

record = {
    "messages": trajectory,
    "tools": TOOL_SCHEMAS,
}

with open(output_file, "a") as f:
    f.write(json.dumps(record) + "\n")

print(f"Saved trajectory to {output_file}")

## Test with a Few More Tickers

In [ ]:
test_tickers = [
    ("AAPL", "competitive position and market share"),
    ("TSLA", "recent news and analyst sentiment"),
    ("JPM", "financial health and balance sheet strength"),
]

for ticker, focus in test_tickers:
    print(f"\n{'='*60}")
    print(f"Researching {ticker} — {focus}")
    print(f"{'='*60}")
    
    traj = run_tool_calling_agent(
        client=client,
        model=MODEL,
        user_prompt=f"Research {ticker} focusing on {focus}.",
        system_prompt=SYSTEM_PROMPT,
    )
    
    n_tools = sum(1 for m in traj if m.get("role") == "tool")
    final = [m for m in traj if m["role"] == "assistant" and not m.get("tool_calls")]
    memo_len = len(final[-1]["content"]) if final else 0
    print(f"  → {n_tools} tool calls, memo length: {memo_len} chars")
    
    # Save
    record = {"messages": traj, "tools": TOOL_SCHEMAS}
    with open(output_file, "a") as f:
        f.write(json.dumps(record) + "\n")